# Sleep Time Prediction Project

This notebook looks at how daily habits connect to sleep time. We also use a simple KNN regression model to predict `SleepTime` from the lifestyle variables in the dataset.


### Introduction
Sleep is really important for physical and mental health, but students can still have a hard time getting enough of it. For this project, we wanted to look at everyday habits like phone use, work hours, caffeine, exercise, reading, and relaxation time to see how they relate to sleep.

The dataset used in this project comes from [Kaggle](https://www.kaggle.com/datasets/govindaramsriram/sleep-time-prediction/data). It has 2,000 rows of daily lifestyle information, including workout time, reading time, phone time, work hours, caffeine intake, relaxation time, and sleep time. The data is synthetic, but it still gives us a useful way to practice looking for patterns and building a prediction model.

Since `SleepTime` is a number measured in hours, this project is a supervised regression problem. We are using the six lifestyle variables to predict a continuous target, which is how many hours someone sleeps.

Our main questions are:

1. Which daily lifestyle habits seem most related to sleep duration?
2. Can these habits predict sleep duration reasonably well?


Team Members:

Hazel Delos Reyes

Praneel Talluri


In [1]:
import pandas as pd
import altair as alt
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

alt.data_transformers.enable("vegafusion")

df = pd.read_csv("data/sleeptime_prediction_dataset.csv")
df.head()


,WorkoutTime,ReadingTime,PhoneTime,WorkHours,CaffeineIntake,RelaxationTime,SleepTime
0,1.12,0.52,3.29,7.89,216.08,0.75,3.45
1,2.85,0.49,4.22,5.03,206.18,0.67,4.88
2,2.20,1.81,4.04,9.23,28.73,0.35,3.61
3,1.80,0.50,1.62,7.68,276.77,1.21,4.94
4,0.47,0.54,1.60,4.94,170.54,0.95,5.50


First, we look at whether the lifestyle habits in the dataset are correlated with sleep time. This helps us see which variables appear most related to the target before building a predictive model.


In [2]:
sleep_df = df.drop(columns=["SleepTime"])
sleep_df.head()

,WorkoutTime,ReadingTime,PhoneTime,WorkHours,CaffeineIntake,RelaxationTime
0,1.12,0.52,3.29,7.89,216.08,0.75
1,2.85,0.49,4.22,5.03,206.18,0.67
2,2.20,1.81,4.04,9.23,28.73,0.35
3,1.80,0.50,1.62,7.68,276.77,1.21
4,0.47,0.54,1.60,4.94,170.54,0.95


The features used in this analysis are `WorkoutTime`, `ReadingTime`, `PhoneTime`, `CaffeineIntake`, `RelaxationTime`, and `WorkHours`. While most of these represent personal habits, `WorkHours` is also included because academic and work demands can directly compete with time available for sleep.


In [3]:
# Analyze Variables by plotting them; Look for correlations between variables; correlation matrix
# plot have captions, are explained with enough information

corr_matrix = df.corr(numeric_only=True).reset_index().melt('index')

col_order = list(df.select_dtypes('number').columns)

alt.Chart(corr_matrix).mark_rect().encode(
    x=alt.X('index:N', title=None, sort=col_order, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('variable:N', title=None, sort=col_order),
    color=alt.Color('value:Q',
        title='Sleep Correlation',
        scale=alt.Scale(scheme='redblue', reverse=True, domain=[-1, 1])
    )
).properties(
    title='Correlation Heatmap of Lifestyle Factors',
    width=400,
    height=400
)


alt.Chart(...)

Here, we use a correlation heatmap to explore which lifestyle habits are related to sleep duration. The color scale makes negative and positive correlations easier to compare across variables.


In [4]:
df.corr(numeric_only=True)['SleepTime'].sort_values()

PhoneTime        -0.322506
WorkHours        -0.298469
CaffeineIntake   -0.076992
ReadingTime       0.067199
RelaxationTime    0.144243
WorkoutTime       0.188368
SleepTime         1.000000
Name: SleepTime, dtype: float64

Phone Time, with a correlation of negative 0.32, and Work Hours, with a correlation of negative 0.30, show the strongest relationships with sleep duration. Both relationships are negative, so in this dataset more phone time and longer work hours tend to go with less sleep.

Workout Time and Relaxation Time have smaller positive relationships with sleep. Reading Time and Caffeine Intake are much weaker, so they do not seem to explain much on their own.

Overall, no single habit tells the whole story. That is why it makes sense to use a model that looks at all six lifestyle variables together.


### Predictive Modeling

Because `SleepTime` is a numerical value, this is a supervised regression problem. We will use the six lifestyle variables to predict sleep duration.

We use KNN regression because it is a simple model from class that predicts using similar observations. KNN depends on distance, so scaling matters. Without scaling, a variable with a much bigger range, like `CaffeineIntake`, could have too much influence just because its numbers are larger.


In [5]:
predictors = [
    "WorkoutTime",
    "ReadingTime",
    "PhoneTime",
    "WorkHours",
    "CaffeineIntake",
    "RelaxationTime",
]

X = df[predictors]
y = df["SleepTime"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=2020
)

knn_pipe = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor()
)

param_grid = {"kneighborsregressor__n_neighbors": range(1, 101)}

knn_tune = GridSearchCV(
    knn_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

knn_tune.fit(X_train, y_train)

best_k = knn_tune.best_params_["kneighborsregressor__n_neighbors"]
cv_rmse = -knn_tune.best_score_

best_k, cv_rmse


(61, np.float64(1.7560649339347605))

The cross validation step compares different values of `k`. The value of `k` tells the model how many nearby observations to use for each prediction. A small `k` can follow random noise too closely, while a very large `k` can smooth out real patterns. We choose the value with the lowest cross validation RMSE.


In [6]:
best_model = knn_tune.best_estimator_
best_model


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kneighborsregressor', KNeighborsRegressor(n_neighbors=61))])

### Model Evaluation

The final step is to test the selected model on data that was not used during training or cross validation. This gives us a better idea of how the model might work on new people.


In [7]:
sleep_predictions = best_model.predict(X_test)

test_rmspe = mean_squared_error(y_test, sleep_predictions) ** 0.5

baseline_predictions = [y_train.mean()] * len(y_test)
baseline_rmspe = mean_squared_error(y_test, baseline_predictions) ** 0.5

model_results = pd.DataFrame({
    "Model": ["KNN regression", "Training mean baseline"],
    "RMSPE (hours)": [test_rmspe, baseline_rmspe]
})

model_results


,Model,RMSPE (hours)
0,KNN regression,1.712595
1,Training mean baseline,1.951720


In [8]:
prediction_preview = pd.DataFrame({
    "Actual SleepTime": y_test.reset_index(drop=True).head(10),
    "Predicted SleepTime": sleep_predictions[:10]
})

prediction_preview


,Actual SleepTime,Predicted SleepTime
0,5.31,5.465410
1,4.47,4.612951
2,4.08,4.543607
3,3.00,4.032623
4,5.50,5.588197
5,5.16,5.303770
6,3.76,3.787049
7,4.39,4.799016
8,4.36,4.530984
9,3.24,3.950656


### Conclusion

The exploratory analysis suggests that no single lifestyle habit completely explains sleep duration. The clearest relationships with `SleepTime` are `PhoneTime` and `WorkHours`, and both are negative. In this dataset, people with more phone time or more work hours tend to sleep less. `WorkoutTime` and `RelaxationTime` have smaller positive relationships with sleep, while `ReadingTime` and `CaffeineIntake` are pretty weak.

The KNN regression model used all six lifestyle variables together to predict sleep duration. Cross validation selected `k = 61`, so each prediction is based on the 61 most similar training observations after scaling the predictors. On the test data, the model has an RMSPE of about 1.71 hours. That means a typical prediction is off by around 1.71 hours of sleep. The model does better than just predicting the training average for everyone, but the error is still large enough that we should not treat the predictions as exact.

Overall, the model finds some useful patterns, especially around phone time and work hours, but sleep is still affected by more than the six variables in this dataset.
